In [1]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format='retina'
%matplotlib inline

In [2]:
import requests
import numpy as np
import pandas as pd
import os,sys,glob
import matplotlib.pyplot as plt
import geopandas as gpd
from mpl_toolkits.axes_grid1 import make_axes_locatable
from shutil import which
import geopandas as gpd
import dask
import zipfile
from pyproj import Proj, transform
import subprocess
from shapely.geometry import Point
import asp_binder_utils as asp_utils
import xyzservices
import rioxarray as rxr
import rasterio
from shapely.geometry import box
from sliderule import earthdata, icesat2, gedi
import contextily as ctx
import sliderule
import xyzservices

In [19]:
%cd /panfs/ccds02/nobackup/people/sbhusha1/pcd/manuscript_prepration/

/panfs/ccds02/nobackup/people/sbhusha1/pcd/manuscript_prepration


## USGS

In [36]:
%cd usgs

/panfs/ccds02/nobackup/people/sbhusha1/pcd/manuscript_prepration/usgs


In [37]:
gf_gedi_stac = gpd.read_file('/home/sbhusha1/notebooks/pcd/20250709_PCD_3dep_gedi_tracks.geojson')
workunit_list = np.unique(gf_gedi_stac.usgs_workunit.values).tolist()

In [38]:
workunit_list

['AZ_PimaCo_2_2021',
 'CA_SanFrancisco_1_B23',
 'CA_YosemiteNP_2019',
 'CO_WestCentral_2019',
 'GA_Central_3_2019',
 'NE_Northeast_Phase2_2_2020',
 'TX_DesertMountains_B1_2018',
 'WI_Brown_2_2020',
 'WY_FEMA_East_B9_2019']

In [8]:
def fetch_gedi(workunit,raster_fn,gpkg_outfn):
    if 'usgs_workunit' in gf_gedi_stac.columns:
        mask = gf_gedi_stac['usgs_workunit'] == workunit
    elif 'neon_id' in gf_gedi_stac.columns:
        mask = gf_gedi_stac['neon_id'] == workunit
    else:
        mask = gf_gedi_stac['ncalm_id'] == workunit
    gedi_ids = [str(value)+'.h5' for value in gf_gedi_stac[mask].gedi_granule_id.values.tolist()]
    print(gedi_ids)
    datetime = [str(np.sort(gf_gedi_stac[mask].date.values)[0]), str(np.sort(gf_gedi_stac[mask].date.values)[-1])]

    da = rxr.open_rasterio(raster_fn, masked=True).squeeze()
    da_bounds = da.rio.bounds()  # (minx, miny, maxx, maxy)
    da_box = box(*da_bounds)
    gf_search = gpd.GeoDataFrame(geometry=[da_box],
                                 columns=['geometry'],
                                 crs = da.rio.crs)
    gf_search = gf_search.to_crs(4326)
    
    poly_sliderule = sliderule.toregion(gf_search)['poly']
    #https://nbviewer.org/github/ICESat2-SlideRule/sliderule-python/blob/main/examples/phoreal.ipynb
    parms = {
        "poly": poly_sliderule,
        "t0": datetime[0],
        "t1": datetime[1],
    }
    
    parms_gedi = {
        "poly": poly_sliderule,
        "t0": datetime[0],
        "t1": datetime[1],
        'anc_fields':['master_frac','energy_total','rh','quality_flag']
    }
    
    
    sliderule.init("slideruleearth.io", verbose=False)
    earthdata.set_max_resources(max_resources=1000)
    
    gf_gedi = gedi.gedi02ap(parms_gedi,
                            resources=gedi_ids)
    
    
    
    
    quality_filter = gf_gedi['quality_flag'] == 1
    gf_gedi_filtered = gf_gedi[quality_filter]
    

    gf_gedi_filtered_proj = gf_gedi_filtered.to_crs(gf_gedi_filtered.estimate_utm_crs())
    gf_gedi_filtered_proj['easting'] = gf_gedi_filtered_proj.geometry.x
    gf_gedi_filtered_proj['northing'] = gf_gedi_filtered_proj.geometry.y
    
    gf_gedi_filtered_proj.to_file(gpkg_outfn,driver='GPKG')

In [40]:
for workunit in workunit_list:
    data_dir = glob.glob(f"{workunit}_*processing*/")[0]
    dtm_fn = glob.glob(os.path.join(data_dir,'*DTM*4*mos.tif'))[0]
    outdir = os.path.join(data_dir,"GEDI")
    if not os.path.exists(outdir):
        os.makedirs(outdir)
    outfn = os.path.join(outdir,f"{workunit}_GEDI_original_pointcloud.gpkg")
    print(dtm_fn,outfn)
    fetch_gedi(workunit,dtm_fn,outfn)

AZ_PimaCo_2_2021_processing/AZ_PimaCo_2_2021-DTM_fill_window_size_4_mos.tif AZ_PimaCo_2_2021_processing/GEDI/AZ_PimaCo_2_2021_GEDI_original_pointcloud.gpkg
['GEDI02_A_2021261011319_O15667_02_T08065_02_003_02_V002.h5', 'GEDI02_A_2021264234230_O15728_02_T10605_02_003_02_V002.h5', 'GEDI02_A_2021285225842_O16053_03_T08636_02_003_02_V002.h5', 'GEDI02_A_2021332204942_O16780_02_T08218_02_003_02_V002.h5']


Warning, this environment is using an outdated client (v5.0.1). The code will run but some functionality supported by the server (v5.3.1) may not be available.


CA_SanFrancisco_1_B23_processing/CA_SanFrancisco_1_B23-DTM_fill_window_size_4_mos.tif CA_SanFrancisco_1_B23_processing/GEDI/CA_SanFrancisco_1_B23_GEDI_original_pointcloud.gpkg
['GEDI02_A_2023028212552_O23387_02_T09397_02_003_02_V002.h5']


Warning, this environment is using an outdated client (v5.0.1). The code will run but some functionality supported by the server (v5.3.0) may not be available.


CA_YosemiteNP_2019_processing/CA_YosemiteNP_2019-DTM_fill_window_size_4_mos.tif CA_YosemiteNP_2019_processing/GEDI/CA_YosemiteNP_2019_GEDI_original_pointcloud.gpkg
['GEDI02_A_2019279200308_O04626_02_T03812_02_003_01_V002.h5', 'GEDI02_A_2019302110619_O04977_02_T00966_02_003_01_V002.h5', 'GEDI02_A_2019309145009_O05088_03_T05331_02_003_01_V002.h5']


...

Warning, this environment is using an outdated client (v5.0.1). The code will run but some functionality supported by the server (v5.3.1) may not be available.


WI_Brown_2_2020_processing/WI_Brown_2_2020-DTM_fill_window_size_4_mos.tif WI_Brown_2_2020_processing/GEDI/WI_Brown_2_2020_GEDI_original_pointcloud.gpkg
['GEDI02_A_2020098225152_O07481_03_T04610_02_003_01_V002.h5', 'GEDI02_A_2020126115921_O07908_03_T04457_02_003_01_V002.h5', 'GEDI02_A_2020130102505_O07969_03_T01764_02_003_01_V002.h5']


Warning, this environment is using an outdated client (v5.0.1). The code will run but some functionality supported by the server (v5.3.0) may not be available.


WY_FEMA_East_B9_2019_processing/WY_FEMA_East_B9_2019-DTM_fill_window_size_4_mos.tif WY_FEMA_East_B9_2019_processing/GEDI/WY_FEMA_East_B9_2019_GEDI_original_pointcloud.gpkg
['GEDI02_A_2019194055411_O03298_02_T00155_02_003_01_V002.h5', 'GEDI02_A_2019200084834_O03393_03_T03035_02_003_02_V002.h5', 'GEDI02_A_2019209000122_O03527_02_T04730_02_003_01_V002.h5', 'GEDI02_A_2019211043557_O03561_03_T00847_02_003_01_V002.h5', 'GEDI02_A_2019219194832_O03695_02_T03965_02_003_02_V002.h5', 'GEDI02_A_2019223180803_O03756_02_T03613_02_003_01_V002.h5', 'GEDI02_A_2019225224231_O03790_03_T02729_02_003_01_V002.h5', 'GEDI02_A_2019242104318_O04046_02_T02343_02_003_02_V002.h5', 'GEDI02_A_2019244151907_O04080_03_T02576_02_003_02_V002.h5', 'GEDI02_A_2019257045811_O04275_02_T00002_02_003_01_V002.h5', 'GEDI02_A_2019261032101_O04336_02_T04883_02_003_01_V002.h5', 'GEDI02_A_2019267062026_O04431_03_T05575_02_003_01_V002.h5']


Warning, this environment is using an outdated client (v5.0.1). The code will run but some functionality supported by the server (v5.3.0) may not be available.


/panfs/ccds02/nobackup/people/sbhusha1/sw/pcd-calval/analysis/gedi-coreg.py --dtm AZ_PimaCo_2_2021_processing/*blur*.tif --gedi AZ_PimaCo_2_2021_processing/GEDI/*GEDI*.gpkg --outdir AZ_PimaCo_2_2021_processing/GEDI/ --asp-dir /panfs/ccds02/nobackup/people/sbhusha1/sw/StereoPipeline-3.6.0-alpha-2025-09-26-x86_64-Linux/bin --verbosem

In [16]:
pwd

'/panfs/ccds02/nobackup/people/sbhusha1/pcd/manuscript_prepration/usgs'

In [41]:
for workunit in workunit_list:
    job_fn = f"slurm_GEDI_coreg_{workunit}.sh"
    gedi_fn = glob.glob(f"{workunit}*processing/GEDI/{workunit}_GEDI_original_pointcloud.gpkg")[0]
    dtm_fn = glob.glob(f"{workunit}*processing/*blur*.tif")[0]
    outdir = os.path.dirname(gedi_fn)
    asp_path = "/panfs/ccds02/nobackup/people/sbhusha1/sw/StereoPipeline-3.6.0-alpha-2025-09-26-x86_64-Linux/bin"
    code_path = "/panfs/ccds02/nobackup/people/sbhusha1/sw/pcd-calval/analysis/gedi-coreg.py"
    with open(job_fn,'w') as f:
        string1 = '#! /bin/bash\n'
        string2 = f"#SBATCH --job-name=GEDI_coreg_{workunit} -t 2:00:00 -c10 --mem=190G\n"
        string3 = f"cd /panfs/ccds02/nobackup/people/sbhusha1/pcd/manuscript_prepration/usgs\n"
        string4 = f"time {code_path} --dtm {dtm_fn} --gedi {gedi_fn} --outdir {outdir} --asp-dir {asp_path} --verbose \n"
        out = string1 + string2 + string3 + string4
        f.write(out)

## NEON

In [20]:
gf_gedi_stac = gpd.read_file('/home/sbhusha1/notebooks/pcd/20250718_PCD_neon_gedi_tracks.geojson')
workunit_list = np.unique(gf_gedi_stac.neon_id.values).tolist()

In [25]:
%cd ../neon

/panfs/ccds02/nobackup/people/sbhusha1/pcd/manuscript_prepration/neon


In [33]:
for workunit in workunit_list:
    data_dir = glob.glob(f"{workunit}_*processing*/")[0]
    dtm_fn = glob.glob(os.path.join(data_dir,'*DTM*4*mos.tif'))[0]
    outdir = os.path.join(data_dir,"GEDI")
    if not os.path.exists(outdir):
        os.makedirs(outdir)
    outfn = os.path.join(outdir,f"{workunit}_GEDI_original_pointcloud.gpkg")
    print(dtm_fn,outfn)
    fetch_gedi(workunit,dtm_fn,outfn)

BART_first_idw_processing/BART_extent-DTM_fill_window_size_4_mos.tif BART_first_idw_processing/GEDI/BART_GEDI_original_pointcloud.gpkg
['GEDI02_A_2019195081130_O03315_03_T05405_02_003_01_V002.h5', 'GEDI02_A_2019210205202_O03556_02_T00995_02_003_01_V002.h5']


Warning, this environment is using an outdated client (v5.0.1). The code will run but some functionality supported by the server (v5.3.0) may not be available.


REDB_first_idw_processing/REDB_extent-DTM_fill_window_size_4_mos.tif REDB_first_idw_processing/GEDI/REDB_GEDI_original_pointcloud.gpkg
['GEDI02_A_2021124140303_O13552_03_T11252_02_003_02_V002.h5', 'GEDI02_A_2021087223155_O12984_02_T08647_02_003_02_V002.h5']


Warning, this environment is using an outdated client (v5.0.1). The code will run but some functionality supported by the server (v5.3.1) may not be available.


WREF_first_idw_processing/WREF_extent-DTM_fill_window_size_4_mos.tif WREF_first_idw_processing/GEDI/WREF_GEDI_original_pointcloud.gpkg
['GEDI02_A_2019206022612_O03482_02_T00370_02_003_01_V002.h5']


Warning, this environment is using an outdated client (v5.0.1). The code will run but some functionality supported by the server (v5.3.1) may not be available.


In [34]:
for workunit in workunit_list:
    job_fn = f"slurm_GEDI_coreg_{workunit}.sh"
    gedi_fn = glob.glob(f"{workunit}*processing/GEDI/{workunit}_GEDI_original_pointcloud.gpkg")[0]
    dtm_fn = glob.glob(f"{workunit}*processing/*blur*.tif")[0]
    outdir = os.path.dirname(gedi_fn)
    asp_path = "/panfs/ccds02/nobackup/people/sbhusha1/sw/StereoPipeline-3.6.0-alpha-2025-09-26-x86_64-Linux/bin"
    code_path = "/panfs/ccds02/nobackup/people/sbhusha1/sw/pcd-calval/analysis/gedi-coreg.py"
    with open(job_fn,'w') as f:
        string1 = '#! /bin/bash\n'
        string2 = f"#SBATCH --job-name=GEDI_coreg_{workunit} -t 2:00:00 -c10 --mem=190G\n"
        string3 = f"cd /panfs/ccds02/nobackup/people/sbhusha1/pcd/manuscript_prepration/neon\n"
        string4 = f"time {code_path} --dtm {dtm_fn} --gedi {gedi_fn} --outdir {outdir} --asp-dir {asp_path} --verbose \n"
        out = string1 + string2 + string3 + string4
        f.write(out)

## NCALM

In [5]:
gf_gedi_stac

,ncalm_id,gedi_granule_id,date,geometry
0,OTLAS.092021.32611.1,GEDI02_A_2020050133846_O06731_02_T04240_02_003...,2020-02-19,"POLYGON ((-116.12955 33.05175, -116.24252 32.9..."
1,OTLAS.092021.32611.1,GEDI02_A_2020046151221_O06670_02_T01394_02_003...,2020-02-15,"POLYGON ((-116.16015 33.07863, -116.30423 32.9..."
2,OTLAS.092021.32611.1,GEDI02_A_2020054120506_O06792_02_T01547_02_003...,2020-02-23,"POLYGON ((-116.01002 33.09856, -116.17718 32.9..."
3,OTLAS.092021.32611.1,GEDI02_A_2020034195258_O06487_02_T01547_02_003...,2020-02-03,"POLYGON ((-115.94227 33.15576, -116.17574 32.9..."
4,OTLAS.092021.32611.1,GEDI02_A_2020026225949_O06365_02_T01394_02_003...,2020-01-26,"POLYGON ((-115.87991 33.31112, -116.29524 32.9..."


In [6]:
gf_gedi_stac = gpd.read_file('/home/sbhusha1/notebooks/pcd/20250722_PCD_ncalm_gedi_tracks.geojson')
workunit_list = np.unique(gf_gedi_stac.ncalm_id.values).tolist()

In [7]:
workunit_list

['OTLAS.092021.32611.1']

In [20]:
%cd ncalm/

/panfs/ccds02/nobackup/people/sbhusha1/pcd/manuscript_prepration/ncalm


In [21]:
workunit_list

['OTLAS.092021.32611.1']

In [22]:
ls

SA_Fault_processing/


In [23]:
for workunit in workunit_list:
    data_dir = glob.glob(f"SA_Fault_processing/")[0]
    dtm_fn = glob.glob(os.path.join(data_dir,'*1m.tif'))[0]
    outdir = os.path.join(data_dir,"GEDI")
    if not os.path.exists(outdir):
        os.makedirs(outdir)
    outfn = os.path.join(outdir,f"SA_Fault_GEDI_original_pointcloud.gpkg")
    print(dtm_fn,outfn)
    fetch_gedi(workunit,dtm_fn,outfn)

SA_Fault_processing/ncalm_sa_fault_dsm_1m.tif SA_Fault_processing/GEDI/SA_Fault_GEDI_original_pointcloud.gpkg
['GEDI02_A_2020050133846_O06731_02_T04240_02_003_01_V002.h5', 'GEDI02_A_2020046151221_O06670_02_T01394_02_003_01_V002.h5', 'GEDI02_A_2020054120506_O06792_02_T01547_02_003_01_V002.h5', 'GEDI02_A_2020034195258_O06487_02_T01547_02_003_01_V002.h5', 'GEDI02_A_2020026225949_O06365_02_T01394_02_003_01_V002.h5']


Warning, this environment is using an outdated client (v5.0.1). The code will run but some functionality supported by the server (v5.3.0) may not be available.
